## Task1: Agent Concepts & Mental Model

An **agent** is an LLM-driven system that can decide what action to take, use tools, inspect the results, and keep going until the task is complete. The important part is that the system can choose the next step instead of following only a fixed sequence.

A **chatbot** is mainly built to have a conversation: it receives a message and generates a response. A chatbot can use tools, but tool use and multi-step decision-making are not required for it to be a chatbot.

A **workflow** is more predictable. The developer defines the steps and their order ahead of time, such as: get the input, call an API, process the result, and display the output. The program follows that path rather than deciding dynamically what to do next.

In simple terms, a **chatbot** mainly responds, a **workflow** mainly follows instructions, and an **agent** can decide what to do next.

**What makes something agentic?** The main characteristics are autonomy in choosing actions, tool use, multi-step execution, and the ability to react to observations or errors and adjust its next step.

### ReAct:

The ReAct pattern can be described simply as **Reason -> Act -> Observe -> repeat**. The user gives a request, the LLM reasons about what needs to happen next, it calls a tool when necessary, observes the result, and then decides whether another step is needed. Once the task is complete, it gives the final answer.

**Flow:** User request -> LLM reasons -> choose an action -> call a tool -> receive the result -> LLM observes the result -> repeat if needed -> final answer.

An agent is overkill when a task can be solved reliably by one prompt, a deterministic Python function, or a fixed sequence of API calls. Agents add complexity, latency, cost, and additional failure modes.


In [ ]:
!pip -q install -U google-genai

import json
from typing import Any, Dict
from google import genai
from google.genai import types
from getpass import getpass

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

if not GEMINI_API_KEY:
    raise RuntimeError("A Gemini API key is required.")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

print("Using model:", MODEL)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.
Using model: gemini-2.5-flash


## Task2: Tool Calling Fundamentals

We define two tools: a calculator and a weather lookup stub. The schemas below use the task's requested name, description, and input_schema structure.

The model uses the tool name, description, and schema to decide when a tool is appropriate and what arguments to send. Clear descriptions reduce wrong tool selection and malformed arguments.


In [ ]:
tools = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression using numbers and +, -, *, /, and parentheses.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A basic arithmetic expression, such as '25 * 4'."
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": "Return stub weather information for a city. This is simulated data, not live weather.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City whose simulated weather should be returned."
                }
            },
            "required": ["city"]
        }
    }
]

print(json.dumps(tools, indent=2))

gemini_tools = [
    types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name=tool["name"],
                description=tool["description"],
                parameters=tool["input_schema"],
            )
            for tool in tools
        ]
    )
]


[
  {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression using numbers and +, -, *, /, and parentheses.",
    "input_schema": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "A basic arithmetic expression, such as '25 * 4'."
        }
      },
      "required": [
        "expression"
      ]
    }
  },
  {
    "name": "get_weather",
    "description": "Return stub weather information for a city. This is simulated data, not live weather.",
    "input_schema": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City whose simulated weather should be returned."
        }
      },
      "required": [
        "city"
      ]
    }
  }
]


In [ ]:
def calculator(expression: str) -> Dict[str, Any]:
    import ast
    import operator as op

    allowed = {
        ast.Add: op.add,
        ast.Sub: op.sub,
        ast.Mult: op.mul,
        ast.Div: op.truediv,
        ast.USub: op.neg,
        ast.UAdd: op.pos,
    }

    def evaluate(node):
        if isinstance(node, ast.Expression):
            return evaluate(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed:
            return allowed[type(node.op)](evaluate(node.operand))
        if isinstance(node, ast.BinOp) and type(node.op) in allowed:
            return allowed[type(node.op)](evaluate(node.left), evaluate(node.right))
        raise ValueError("Unsupported expression")

    return {"result": evaluate(ast.parse(expression, mode="eval"))}


def get_weather(city: str) -> Dict[str, Any]:
    # Simulated weather data
    weather = {
        "Lahore": {"temperature_c": 34, "condition": "Sunny"},
        "Karachi": {"temperature_c": 29, "condition": "Partly cloudy"},
        "Islamabad": {"temperature_c": 27, "condition": "Cloudy"},
        "Faisalabad": {"temperature_c": 33, "condition": "Sunny"},
    }
    return weather.get(city, {"error": f"No stub weather data for {city}"})


TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_weather": get_weather,
}

print("Tools ready:", list(TOOL_FUNCTIONS))


Tools ready: ['calculator', 'get_weather']


### Single tool request + manual tool result

The model chooses the tool. We manually execute the Python function and then send a function-response block back to Gemini. This makes the tool-calling mechanism visible instead of hiding it behind an automatic agent framework.


In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is 25 multiplied by 8?",
    config=types.GenerateContentConfig(
        tools=gemini_tools,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)

tool_calls = [
    part.function_call
    for part in response.candidates[0].content.parts
    if part.function_call
]

if not tool_calls:
    print("The model returned text instead of a tool call:")
    print(response.text)
else:
    tool_call = tool_calls[0]
    print("Tool selected:", tool_call.name)
    print("Arguments:", dict(tool_call.args))


Tool selected: calculator
Arguments: {'expression': '25 * 8'}


In [ ]:
if tool_calls:
    result = TOOL_FUNCTIONS[tool_call.name](**dict(tool_call.args))

    tool_result_part = types.Part.from_function_response(
        name=tool_call.name,
        response=result,
    )

    print("Executed result:", result)
    print("Function response part:", tool_result_part)


Executed result: {'result': 200}
Function response part: media_resolution=None code_execution_result=None executable_code=None file_data=None function_call=None function_response=FunctionResponse(
  name='calculator',
  response={
    'result': 200
  }
) inline_data=None text=None thought=None thought_signature=None video_metadata=None tool_call=None tool_response=None part_metadata=None audio_transcription=None media_processing=None


## Task3: Build a Minimal Agent Loop

The loop:

1. Sends the user request to Gemini.
2. Checks whether Gemini returned one or more function_call parts.
3. Executes the requested Python tools manually.
4. Appends the function responses to conversation memory.
5. Sends the updated history back to Gemini.
6. Stops when Gemini returns normal text.
7. Stops with an error if max_iterations is reached.

Gemini calls this feature function calling. Conceptually, it implements the same core loop as ReAct: Reason → Act → Observe → repeat


In [ ]:
def run_agent(user_prompt: str, max_iterations: int = 6, verbose: bool = True):
    # Conversation memory: everything the model needs to see from this run.
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_prompt)]
        )
    ]

    # Working state: application-level information tracked during execution.
    state = {
        "iterations": 0,
        "tools_used": [],
        "observations": [],
    }

    while state["iterations"] < max_iterations:
        state["iterations"] += 1

        if verbose:
            print(f"\n[Agent] Iteration {state['iterations']}")

        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=types.GenerateContentConfig(
                tools=gemini_tools,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
            ),
        )

        model_content = response.candidates[0].content
        contents.append(model_content)

        # Gemini may return multiple function calls in one model response.
        tool_calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        if not tool_calls:
            final_text = "\n".join(
                part.text for part in model_content.parts if part.text
            )

            if verbose:
                print("[Agent] Final answer:\n", final_text)

            return final_text, state, contents

        function_response_parts = []

        for call in tool_calls:
            tool_name = call.name
            arguments = dict(call.args)
            state["tools_used"].append(tool_name)

            if verbose:
                print(f"[Tool call] {tool_name}({arguments})")

            try:
                if tool_name not in TOOL_FUNCTIONS:
                    raise ValueError(f"Unknown tool: {tool_name}")

                result = TOOL_FUNCTIONS[tool_name](**arguments)

            except Exception as exc:
                result = {"error": str(exc)}

            state["observations"].append(result)

            if verbose:
                print("[Observation]", result)

            function_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response=result,
                )
            )

        # Send tool observations back to Gemini as the next message.
        contents.append(
            types.Content(
                role="user",
                parts=function_response_parts
            )
        )

    raise RuntimeError(
        f"Agent stopped after max_iterations={max_iterations}"
    )


In [ ]:
final_answer, state, messages = run_agent(
    "Look up the weather in Lahore and Karachi and tell me which city is warmer and by how many degrees.",
    max_iterations=6,
)

print("\nState:", state)



[Agent] Iteration 1
[Tool call] get_weather({'city': 'Lahore'})
[Observation] {'temperature_c': 34, 'condition': 'Sunny'}
[Tool call] get_weather({'city': 'Karachi'})
[Observation] {'temperature_c': 29, 'condition': 'Partly cloudy'}

[Agent] Iteration 2
[Agent] Final answer:
 Lahore is warmer than Karachi by 5 degrees Celsius. It is 34°C in Lahore and 29°C in Karachi.

State: {'iterations': 2, 'tools_used': ['get_weather', 'get_weather'], 'observations': [{'temperature_c': 34, 'condition': 'Sunny'}, {'temperature_c': 29, 'condition': 'Partly cloudy'}]}


## Task4: Memory & State Handling

**Conversation memory** is the message history passed back to Gemini. It preserves the original request, model function calls, and tool observations so the model can understand what has already happened.

**Working memory/state** is application-level information tracked during execution, such as the iteration count, tools used, observations, intermediate values, or task status.

In this implementation, contents is conversation memory and state is working state. Logging prints each iteration, tool call, and observation. This is a useful debugging habit when working with any agent framework.


## Task5: Failure Modes & Guardrails

We deliberately test an unsupported city. The weather tool returns a structured error instead of inventing data, and that error is sent back to Gemini as an observation so the model can decide how to respond.


In [ ]:
failure_answer, failure_state, _ = run_agent(
    "What is the weather in Atlantis?",
    max_iterations=4,
)

print("\nFailure-test state:", failure_state)



[Agent] Iteration 1
[Tool call] get_weather({'city': 'Atlantis'})
[Observation] {'error': 'No stub weather data for Atlantis'}

[Agent] Iteration 2
[Agent] Final answer:
 There is no weather information for Atlantis.

Failure-test state: {'iterations': 2, 'tools_used': ['get_weather'], 'observations': [{'error': 'No stub weather data for Atlantis'}]}


### Failure-mode documentation

Here are the main failure modes I considered:

1. **Infinite loop:** The agent may keep calling tools without reaching a useful answer. **Mitigation:** set a max_iterations limit.

2. **Wrong tool:** The model may choose a tool that does not fit the request. **Mitigation:** give each tool a clear, specific description and define when it should be used.

3. **Wrong arguments:** The model may send an invalid or missing argument. **Mitigation:** use a JSON schema and validate arguments in the Python tool before executing them.

4. **Tool/API error:** A tool may fail or return an error. **Mitigation:** catch exceptions and return a structured error observation to the model.

5. **Missing capability:** The user may request something for which no tool exists. **Mitigation:** fail gracefully and make the agent's available capabilities explicit.

6. **Hallucinated result:** The model may claim that it obtained a result even though a tool was never executed. **Mitigation:** require actual tool execution and log every tool call and observation.

### Why do frameworks exist?

The raw implementation exposes the core mechanism, but production agents need reusable infrastructure for tool management, state, retries, branching, persistence, observability, structured outputs, human approval, and multi-agent coordination. Frameworks such as LangChain, LangGraph, and CrewAI provide abstractions around these patterns so developers do not rebuild the same orchestration machinery for every project.
